In [ ]:
"""
Phishing URL Detection System — Flask Backend
=============================================
Scoring approach:
  Each heuristic adds a risk score (0–100 total possible penalty points).
  A final score is clamped to [0, 100]; higher = more suspicious.

  Risk bands:
    0–29   → Safe (green)
   30–59   → Suspicious (yellow)
   60–100  → Likely Phishing (red)

Heuristics and their weights are documented in SCORING.md.
No URLs or user data are stored, logged to disk, or sent externally.
"""

from flask import Flask, render_template, request, jsonify
import re
import math
from urllib.parse import urlparse, unquote

app = Flask(__name__)


# ---------------------------------------------------------------------------
# Scoring heuristics
# Each function returns (score_delta: int, reason: str | None)
# ---------------------------------------------------------------------------

SUSPICIOUS_TLDS = {
    ".xyz", ".top", ".club", ".work", ".gq", ".ml", ".cf", ".tk", ".ga",
    ".pw", ".click", ".link", ".loan", ".win", ".download", ".accountant",
    ".stream", ".faith", ".review", ".party", ".science", ".trade",
}

TRUSTED_BRANDS = [
    "paypal", "apple", "google", "microsoft", "amazon", "facebook",
    "instagram", "twitter", "netflix", "ebay", "linkedin", "dropbox",
    "bankofamerica", "chase", "wellsfargo", "citibank", "irs", "gov",
]

PHISHING_KEYWORDS = [
    "login", "signin", "verify", "account", "update", "secure", "confirm",
    "banking", "password", "credential", "wallet", "support", "helpdesk",
    "billing", "invoice", "alert", "suspended", "unlock", "validate",
]


def check_url_length(url: str) -> tuple[int, str | None]:
    """Long URLs obscure the true destination. >75 chars adds risk."""
    length = len(url)
    if length > 100:
        return 15, f"URL is very long ({length} chars)"
    if length > 75:
        return 8, f"URL is long ({length} chars)"
    return 0, None


def check_ip_address(parsed) -> tuple[int, str | None]:
    """Legitimate sites use domain names, not bare IP addresses."""
    host = parsed.hostname or ""
    ipv4 = re.match(r"^\d{1,3}(\.\d{1,3}){3}$", host)
    if ipv4:
        return 20, "Host is a raw IP address (no domain name)"
    return 0, None


def check_suspicious_tld(parsed) -> tuple[int, str | None]:
    """Several cheap/free TLDs are disproportionately used in phishing."""
    host = (parsed.hostname or "").lower()
    for tld in SUSPICIOUS_TLDS:
        if host.endswith(tld):
            return 15, f"Domain uses a high-risk TLD ({tld})"
    return 0, None


def check_subdomain_depth(parsed) -> tuple[int, str | None]:
    """Phishers add sub-domains to fake legitimacy (e.g. paypal.verify.evil.com)."""
    host = (parsed.hostname or "").lower()
    parts = [p for p in host.split(".") if p]
    depth = len(parts) - 2  # subtract SLD + TLD
    if depth >= 3:
        return 15, f"Excessive sub-domain depth ({depth} levels)"
    if depth == 2:
        return 7, f"Multiple sub-domains ({depth} levels)"
    return 0, None


def check_brand_in_subdomain(parsed) -> tuple[int, str | None]:
    """A trusted brand name in a sub-domain but not in the SLD is classic phishing."""
    host = (parsed.hostname or "").lower()
    parts = host.split(".")
    if len(parts) < 3:
        return 0, None
    # Everything except the last two parts = sub-domains
    subdomains = ".".join(parts[:-2])
    sld = parts[-2]
    for brand in TRUSTED_BRANDS:
        if brand in subdomains and brand not in sld:
            return 20, f"Trusted brand '{brand}' appears in sub-domain only"
    return 0, None


def check_https(parsed) -> tuple[int, str | None]:
    """Plain HTTP provides no transport security."""
    if parsed.scheme == "http":
        return 8, "No HTTPS — connection is unencrypted"
    return 0, None


def check_special_chars(url: str) -> tuple[int, str | None]:
    """@ in URLs redirects; multiple dashes or encodings are evasion tricks."""
    score = 0
    reasons = []
    if "@" in url:
        score += 15
        reasons.append("URL contains '@' (can redirect to different host)")
    double_slash = re.findall(r"(?<!:)//", url)
    if double_slash:
        score += 10
        reasons.append("URL contains '//' outside of protocol")
    dash_count = url.count("-")
    if dash_count >= 4:
        score += 5
        reasons.append(f"URL contains many hyphens ({dash_count})")
    return score, "; ".join(reasons) if reasons else None


def check_phishing_keywords(url: str) -> tuple[int, str | None]:
    """Security-sensitive words in the path/query are common lures."""
    lower = url.lower()
    found = [kw for kw in PHISHING_KEYWORDS if kw in lower]
    if len(found) >= 3:
        return 15, f"Multiple phishing keywords: {', '.join(found[:5])}"
    if found:
        return 7, f"Phishing keyword(s) detected: {', '.join(found[:3])}"
    return 0, None


def check_url_encoding(url: str) -> tuple[int, str | None]:
    """Heavy percent-encoding is used to obscure malicious paths."""
    encoded_count = len(re.findall(r"%[0-9A-Fa-f]{2}", url))
    if encoded_count >= 5:
        return 10, f"Heavy URL encoding ({encoded_count} encoded chars)"
    return 0, None


def check_port(parsed) -> tuple[int, str | None]:
    """Non-standard ports are unusual for legitimate web services."""
    port = parsed.port
    if port and port not in (80, 443, 8080, 8443):
        return 10, f"Non-standard port ({port})"
    return 0, None


def check_path_depth(parsed) -> tuple[int, str | None]:
    """Very deep paths can indicate redirect chains."""
    path = parsed.path or ""
    depth = len([p for p in path.split("/") if p])
    if depth >= 6:
        return 8, f"Very deep URL path ({depth} segments)"
    return 0, None


# ---------------------------------------------------------------------------
# Aggregator
# ---------------------------------------------------------------------------

HEURISTICS = [
    check_url_length,
    check_ip_address,
    check_suspicious_tld,
    check_subdomain_depth,
    check_brand_in_subdomain,
    check_https,
    check_special_chars,
    check_phishing_keywords,
    check_url_encoding,
    check_port,
    check_path_depth,
]


def analyse_url(url: str) -> dict:
    """
    Run all heuristics and return a result dict.
    score: 0 (safe) – 100 (certain phishing).
    """
    url = url.strip()
    if not url:
        return {"error": "No URL provided"}

    # Normalise — add scheme if missing so urlparse works correctly
    if not re.match(r"^[a-zA-Z][a-zA-Z\d+\-.]*://", url):
        url = "http://" + url

    try:
        parsed = urlparse(url)
    except Exception:
        return {"error": "Could not parse URL"}

    total_score = 0
    flags = []

    for heuristic in HEURISTICS:
        try:
            # Functions that only need the parsed object
            import inspect
            sig = inspect.signature(heuristic)
            param = list(sig.parameters.keys())[0]
            delta, reason = heuristic(parsed) if param == "parsed" else heuristic(url)
        except Exception:
            continue

        if delta > 0:
            total_score += delta
            if reason:
                flags.append(reason)

    total_score = min(total_score, 100)

    if total_score < 30:
        verdict = "Safe"
        band = "safe"
    elif total_score < 60:
        verdict = "Suspicious"
        band = "suspicious"
    else:
        verdict = "Likely Phishing"
        band = "danger"

    improvements = _suggest_improvements(band, flags)

    return {
        "url": url,
        "score": total_score,
        "verdict": verdict,
        "band": band,
        "flags": flags,
        "improvements": improvements,
    }


def _suggest_improvements(band: str, flags: list[str]) -> list[str]:
    """Return actionable suggestions for the URL owner (not the checker user)."""
    suggestions = []
    flag_text = " ".join(flags).lower()

    if "http" in flag_text and "https" not in flag_text:
        suggestions.append("Enable HTTPS with a valid TLS certificate.")
    if "ip address" in flag_text:
        suggestions.append("Use a registered domain name instead of a bare IP.")
    if "sub-domain" in flag_text:
        suggestions.append("Reduce sub-domain nesting to one level where possible.")
    if "tld" in flag_text:
        suggestions.append("Register on a well-established TLD (.com, .org, .net).")
    if "keyword" in flag_text:
        suggestions.append("Avoid security-sensitive words in URLs unless necessary.")
    if "long" in flag_text:
        suggestions.append("Shorten the URL; very long URLs reduce user trust.")
    if "encoding" in flag_text:
        suggestions.append("Avoid heavy percent-encoding in publicly shared URLs.")
    if "port" in flag_text:
        suggestions.append("Serve on standard ports (80/443) for public-facing URLs.")
    if not suggestions and band != "safe":
        suggestions.append("Review the flagged issues and restructure the URL path.")
    return suggestions


# ---------------------------------------------------------------------------
# Routes
# ---------------------------------------------------------------------------

@app.route("/")
def index():
    return render_template("index.html")


@app.route("/api/analyse", methods=["POST"])
def api_analyse():
    data = request.get_json(silent=True) or {}
    url = data.get("url", "")
    result = analyse_url(url)
    return jsonify(result)


if __name__ == "__main__":
    app.run(debug=True)

In [ ]:
# tests/__init__.py
# Marks the tests directory as a Python package.

"""
tests/test_app.py — Automated Tests for PhishGuard
====================================================
Covers:
  • Individual heuristic functions
  • The analyse_url aggregator
  • Flask API endpoint (/api/analyse)
  • Edge cases (empty input, malformed URLs, boundary scores)

Run with:
    pytest tests/test_app.py -v
"""

import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import pytest
from urllib.parse import urlparse

from app import (
    app,
    analyse_url,
    check_url_length,
    check_ip_address,
    check_suspicious_tld,
    check_subdomain_depth,
    check_brand_in_subdomain,
    check_https,
    check_special_chars,
    check_phishing_keywords,
    check_url_encoding,
    check_port,
    check_path_depth,
)


# ── Fixtures ─────────────────────────────────────────────────────────────────

@pytest.fixture
def client():
    """Flask test client."""
    app.config["TESTING"] = True
    with app.test_client() as c:
        yield c


# ── Helper ───────────────────────────────────────────────────────────────────

def parsed(url: str):
    return urlparse(url)


# ═══════════════════════════════════════════════════════════════════════════
# Individual heuristic tests
# ═══════════════════════════════════════════════════════════════════════════

class TestCheckUrlLength:
    def test_short_url_no_penalty(self):
        score, reason = check_url_length("http://example.com")
        assert score == 0 and reason is None

    def test_moderately_long_url(self):
        url = "http://example.com/" + "a" * 60   # ~80 chars
        score, reason = check_url_length(url)
        assert score == 8

    def test_very_long_url(self):
        url = "http://example.com/" + "a" * 90   # ~110 chars
        score, reason = check_url_length(url)
        assert score == 15

    def test_exactly_75_chars_no_penalty(self):
        url = "h" * 75
        score, _ = check_url_length(url)
        assert score == 0

    def test_exactly_76_chars_moderate_penalty(self):
        url = "h" * 76
        score, _ = check_url_length(url)
        assert score == 8


class TestCheckIpAddress:
    def test_domain_no_penalty(self):
        score, _ = check_ip_address(parsed("http://example.com"))
        assert score == 0

    def test_ipv4_penalised(self):
        score, reason = check_ip_address(parsed("http://192.168.1.1/login"))
        assert score == 20
        assert reason is not None

    def test_partial_ip_no_penalty(self):
        score, _ = check_ip_address(parsed("http://192.168.domain.com"))
        assert score == 0


class TestCheckSuspiciousTld:
    def test_common_tld_safe(self):
        score, _ = check_suspicious_tld(parsed("http://google.com"))
        assert score == 0

    def test_xyz_tld_penalised(self):
        score, reason = check_suspicious_tld(parsed("http://login-paypal.xyz"))
        assert score == 15

    def test_tk_tld_penalised(self):
        score, _ = check_suspicious_tld(parsed("http://free-stuff.tk"))
        assert score == 15

    def test_org_tld_safe(self):
        score, _ = check_suspicious_tld(parsed("http://wikipedia.org"))
        assert score == 0


class TestCheckSubdomainDepth:
    def test_no_subdomain(self):
        score, _ = check_subdomain_depth(parsed("http://example.com"))
        assert score == 0

    def test_single_subdomain_www(self):
        score, _ = check_subdomain_depth(parsed("http://www.example.com"))
        assert score == 0

    def test_two_level_subdomain(self):
        score, reason = check_subdomain_depth(parsed("http://a.b.example.com"))
        assert score == 7

    def test_three_level_subdomain(self):
        score, reason = check_subdomain_depth(parsed("http://a.b.c.example.com"))
        assert score == 15


class TestCheckBrandInSubdomain:
    def test_no_brand(self):
        score, _ = check_brand_in_subdomain(parsed("http://mysite.com"))
        assert score == 0

    def test_brand_in_sld_only_no_penalty(self):
        # paypal is in the SLD, not only in subdomain
        score, _ = check_brand_in_subdomain(parsed("http://www.paypal.com"))
        assert score == 0

    def test_brand_only_in_subdomain(self):
        score, reason = check_brand_in_subdomain(
            parsed("http://paypal.secure-login.com")
        )
        assert score == 20
        assert "paypal" in reason.lower()

    def test_google_in_subdomain(self):
        score, _ = check_brand_in_subdomain(
            parsed("http://google.accounts.evil.net")
        )
        assert score == 20


class TestCheckHttps:
    def test_https_no_penalty(self):
        score, _ = check_https(parsed("https://example.com"))
        assert score == 0

    def test_http_penalised(self):
        score, reason = check_https(parsed("http://example.com"))
        assert score == 8
        assert reason is not None


class TestCheckSpecialChars:
    def test_clean_url_no_penalty(self):
        score, _ = check_special_chars("https://example.com/page")
        assert score == 0

    def test_at_sign_penalised(self):
        score, reason = check_special_chars("http://user@evil.com")
        assert score >= 15

    def test_double_slash_penalised(self):
        score, reason = check_special_chars("http://evil.com//redirect")
        assert score >= 10

    def test_many_hyphens_penalised(self):
        score, _ = check_special_chars("http://my-long-hyphen-test-site.com")
        assert score >= 5

    def test_combined_penalties(self):
        score, _ = check_special_chars("http://user@site.com//path-a-b-c-d")
        # @ (15) + double-slash (10) + hyphens (5) = 30
        assert score >= 25


class TestCheckPhishingKeywords:
    def test_no_keywords(self):
        score, _ = check_phishing_keywords("https://example.com/products")
        assert score == 0

    def test_one_keyword(self):
        score, reason = check_phishing_keywords("https://example.com/login")
        assert score == 7

    def test_three_keywords(self):
        score, _ = check_phishing_keywords(
            "https://example.com/secure/account/verify"
        )
        assert score == 15

    def test_keywords_case_insensitive(self):
        score, _ = check_phishing_keywords("https://example.com/LOGIN/VERIFY/SECURE")
        assert score == 15


class TestCheckUrlEncoding:
    def test_no_encoding(self):
        score, _ = check_url_encoding("https://example.com/path")
        assert score == 0

    def test_light_encoding_no_penalty(self):
        score, _ = check_url_encoding("https://example.com/%20foo%20bar")
        assert score == 0   # only 2 encoded chars

    def test_heavy_encoding_penalised(self):
        encoded = "https://example.com/%2F%2F%65%76%69%6C%2E%63%6F%6D"
        score, _ = check_url_encoding(encoded)
        assert score == 10


class TestCheckPort:
    def test_standard_https_port(self):
        score, _ = check_port(parsed("https://example.com:443/page"))
        assert score == 0

    def test_standard_http_port(self):
        score, _ = check_port(parsed("http://example.com:80/page"))
        assert score == 0

    def test_non_standard_port(self):
        score, reason = check_port(parsed("http://example.com:8888/admin"))
        assert score == 10

    def test_no_port_specified(self):
        score, _ = check_port(parsed("https://example.com/page"))
        assert score == 0


class TestCheckPathDepth:
    def test_shallow_path(self):
        score, _ = check_path_depth(parsed("https://example.com/a/b"))
        assert score == 0

    def test_deep_path(self):
        score, _ = check_path_depth(parsed("https://example.com/a/b/c/d/e/f/g"))
        assert score == 8

    def test_exactly_6_segments(self):
        score, _ = check_path_depth(parsed("https://example.com/a/b/c/d/e/f"))
        assert score == 8


# ═══════════════════════════════════════════════════════════════════════════
# analyse_url aggregator tests
# ═══════════════════════════════════════════════════════════════════════════

class TestAnalyseUrl:
    def test_empty_string_returns_error(self):
        result = analyse_url("")
        assert "error" in result

    def test_safe_url(self):
        result = analyse_url("https://www.google.com")
        assert result["score"] < 30
        assert result["band"] == "safe"

    def test_clearly_phishing_url(self):
        # IP (20) + no-HTTPS (8) + keywords ≥3 (15) + path depth (8) = 51 → suspicious
        # Use a URL that also hits subdomain + TLD to push into danger territory
        url = "http://paypal.secure-verify.xyz:9999/login/verify/account/billing"
        result = analyse_url(url)
        assert result["score"] >= 60
        assert result["band"] == "danger"

    def test_suspicious_url(self):
        url = "http://paypal-secure-login.xyz/account/verify"
        result = analyse_url(url)
        assert result["score"] >= 30

    def test_score_capped_at_100(self):
        # Pile on every possible signal
        url = (
            "http://user@paypal.secure-login-verify-account.xyz:9999"
            "/login/secure/verify/account/update/billing/"
            + "%2F" * 10
            + "-".join(["x"] * 10)
        )
        result = analyse_url(url)
        assert result["score"] <= 100

    def test_result_has_all_keys(self):
        result = analyse_url("https://example.com")
        for key in ("url", "score", "verdict", "band", "flags", "improvements"):
            assert key in result

    def test_url_without_scheme_accepted(self):
        result = analyse_url("example.com")
        assert "error" not in result

    def test_verdict_matches_band(self):
        result = analyse_url("https://www.github.com")
        band_verdicts = {"safe": "Safe", "suspicious": "Suspicious", "danger": "Likely Phishing"}
        assert result["verdict"] == band_verdicts[result["band"]]


# ═══════════════════════════════════════════════════════════════════════════
# Flask API endpoint tests
# ═══════════════════════════════════════════════════════════════════════════

class TestApiEndpoint:
    def test_get_root_returns_200(self, client):
        resp = client.get("/")
        assert resp.status_code == 200
        assert b"PhishGuard" in resp.data

    def test_api_analyse_valid_url(self, client):
        resp = client.post(
            "/api/analyse",
            json={"url": "https://www.example.com"},
        )
        assert resp.status_code == 200
        data = resp.get_json()
        assert "score" in data
        assert "band" in data

    def test_api_analyse_missing_url(self, client):
        resp = client.post("/api/analyse", json={})
        assert resp.status_code == 200
        data = resp.get_json()
        assert "error" in data

    def test_api_analyse_phishing_url(self, client):
        resp = client.post(
            "/api/analyse",
            json={"url": "http://192.168.1.1/secure/login/verify"},
        )
        assert resp.status_code == 200
        data = resp.get_json()
        assert data["band"] in ("suspicious", "danger")

    def test_api_content_type_json(self, client):
        resp = client.post(
            "/api/analyse",
            json={"url": "https://example.com"},
        )
        assert "application/json" in resp.content_type

    def test_api_no_json_body(self, client):
        resp = client.post("/api/analyse", data="not json",
                           content_type="text/plain")
        assert resp.status_code == 200
        data = resp.get_json()
        assert "error" in data

    def test_api_very_long_url(self, client):
        url = "http://example.com/" + "a" * 200
        resp = client.post("/api/analyse", json={"url": url})
        assert resp.status_code == 200
        data = resp.get_json()
        assert data["score"] > 0


# ═══════════════════════════════════════════════════════════════════════════
# Edge cases
# ═══════════════════════════════════════════════════════════════════════════

class TestEdgeCases:
    def test_whitespace_only_url(self):
        result = analyse_url("   ")
        assert "error" in result

    def test_url_with_unicode(self):
        result = analyse_url("https://müller.de")
        assert "error" not in result

    def test_ftp_scheme(self):
        result = analyse_url("ftp://files.example.com/data")
        assert "error" not in result

    def test_no_path_url(self):
        result = analyse_url("https://example.com")
        assert result["score"] >= 0

    def test_flags_is_list(self):
        result = analyse_url("https://example.com")
        assert isinstance(result["flags"], list)

    def test_improvements_is_list(self):
        result = analyse_url("https://example.com")
        assert isinstance(result["improvements"], list)

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Phishing URL Detector</title>
  <link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}" />
</head>
<body>
  <div class="page-wrapper">

    <!-- ── Header ── -->
    <header class="site-header">
      <div class="logo">
        <span class="logo-icon" aria-hidden="true">🛡️</span>
        <span class="logo-text">PhishGuard</span>
      </div>
      <p class="tagline">Instant heuristic analysis of any URL</p>
    </header>

    <!-- ── Main card ── -->
    <main class="card" id="main-card">
      <form id="url-form" novalidate autocomplete="off">
        <label class="input-label" for="url-input">Enter a URL to analyse</label>
        <div class="input-row">
          <input
            id="url-input"
            class="url-input"
            type="url"
            placeholder="https://example.com/path?query=value"
            aria-label="URL to analyse"
            spellcheck="false"
            required
          />
          <button class="btn-analyse" type="submit" id="btn-analyse">
            <span class="btn-text">Analyse</span>
            <span class="btn-spinner" aria-hidden="true" hidden></span>
          </button>
        </div>
        <p class="input-hint" id="input-error" role="alert" aria-live="polite"></p>
      </form>

      <!-- ── Result panel (hidden until a result arrives) ── -->
      <section class="result-panel" id="result-panel" hidden aria-live="polite">

        <!-- Score ring -->
        <div class="score-section">
          <div class="score-ring-wrap">
            <svg class="score-ring" viewBox="0 0 120 120" aria-hidden="true">
              <circle class="ring-bg"    cx="60" cy="60" r="50" />
              <circle class="ring-fill"  cx="60" cy="60" r="50" id="ring-fill" />
            </svg>
            <div class="score-centre">
              <span class="score-value" id="score-value">0</span>
              <span class="score-label">/ 100</span>
            </div>
          </div>
          <div class="verdict-block">
            <span class="verdict-badge" id="verdict-badge">Safe</span>
            <p class="verdict-url" id="verdict-url"></p>
          </div>
        </div>

        <!-- Flags list -->
        <div class="flags-section" id="flags-section">
          <h2 class="section-title">Risk Signals</h2>
          <ul class="flags-list" id="flags-list"></ul>
          <p class="no-flags" id="no-flags" hidden>✅ No risk signals detected.</p>
        </div>

        <!-- Improvements -->
        <div class="improvements-section" id="improvements-section">
          <h2 class="section-title">Improvement Suggestions</h2>
          <ul class="improvements-list" id="improvements-list"></ul>
          <p class="no-flags" id="no-improvements" hidden>✅ No improvements needed.</p>
        </div>

        <!-- Scoring legend -->
        <details class="scoring-legend">
          <summary>How is the score calculated?</summary>
          <div class="legend-body">
            <p>
              Each heuristic check adds penalty points. Points are summed and capped
              at <strong>100</strong>.
            </p>
            <table class="legend-table">
              <thead>
                <tr><th>Check</th><th>Max penalty</th><th>Reason</th></tr>
              </thead>
              <tbody>
                <tr><td>Raw IP address</td><td>20</td><td>Legitimate sites use domain names</td></tr>
                <tr><td>Brand in sub-domain only</td><td>20</td><td>Classic impersonation tactic</td></tr>
                <tr><td>URL contains @</td><td>15</td><td>Redirects browser to a different host</td></tr>
                <tr><td>Suspicious TLD</td><td>15</td><td>Free/cheap TLDs over-represented in phishing</td></tr>
                <tr><td>Sub-domain depth ≥ 3</td><td>15</td><td>Hides real domain deep in the hostname</td></tr>
                <tr><td>Phishing keywords (≥ 3)</td><td>15</td><td>Login, verify, secure, etc.</td></tr>
                <tr><td>URL length &gt; 100 chars</td><td>15</td><td>Obscures the real destination</td></tr>
                <tr><td>Double slash in path</td><td>10</td><td>Evasion / redirect trick</td></tr>
                <tr><td>Heavy URL encoding</td><td>10</td><td>Hides malicious path segments</td></tr>
                <tr><td>Non-standard port</td><td>10</td><td>Unusual for legitimate web services</td></tr>
                <tr><td>No HTTPS</td><td>8</td><td>Unencrypted, easier to intercept</td></tr>
                <tr><td>URL length 75–100 chars</td><td>8</td><td>Moderately long</td></tr>
                <tr><td>Deep path (≥ 6 segments)</td><td>8</td><td>May indicate redirect chains</td></tr>
                <tr><td>Sub-domain depth = 2</td><td>7</td><td>Slightly elevated nesting</td></tr>
                <tr><td>Phishing keyword (1–2)</td><td>7</td><td>One or two lure words</td></tr>
                <tr><td>Many hyphens (≥ 4)</td><td>5</td><td>Hyphen-spam to mimic real domains</td></tr>
              </tbody>
            </table>
            <p class="legend-bands">
              <span class="band-safe">0–29 → Safe</span>
              &nbsp;|&nbsp;
              <span class="band-suspicious">30–59 → Suspicious</span>
              &nbsp;|&nbsp;
              <span class="band-danger">60–100 → Likely Phishing</span>
            </p>
          </div>
        </details>

      </section>
    </main>

    <!-- ── Privacy notice ── -->
    <p class="privacy-note">
      🔒 All analysis is done server-side in memory. No URLs are stored, logged, or sent to third parties.
    </p>

    <footer class="site-footer">PhishGuard · Heuristic-based URL Safety Checker</footer>
  </div>

  <script src="{{ url_for('static', filename='js/main.js') }}"></script>
</body>
</html>

In [ ]:
/* ==========================================================================
   PhishGuard — Main Stylesheet
   ========================================================================== */

/* ── Reset / base ── */
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

:root {
  --bg:            #f0f2f5;
  --surface:       #ffffff;
  --border:        #e2e6ea;
  --text:          #1c2333;
  --muted:         #5a6272;
  --accent:        #3b6fd4;
  --accent-hover:  #2c58b0;

  --safe:          #1a9e5c;
  --safe-bg:       #e6f7ef;
  --safe-ring:     #1a9e5c;

  --suspicious:    #c97c00;
  --suspicious-bg: #fff8e6;
  --suspicious-ring:#e5920a;

  --danger:        #c0392b;
  --danger-bg:     #fdecea;
  --danger-ring:   #e53935;

  --radius:        10px;
  --shadow:        0 2px 12px rgba(0,0,0,0.08);
  --transition:    0.22s ease;
}

html { font-size: 15px; }

body {
  font-family: -apple-system, "Segoe UI", system-ui, sans-serif;
  background: var(--bg);
  color: var(--text);
  line-height: 1.6;
  min-height: 100vh;
  display: flex;
  align-items: flex-start;
  justify-content: center;
  padding: 2rem 1rem 4rem;
}

/* ── Page wrapper ── */
.page-wrapper {
  width: 100%;
  max-width: 720px;
  display: flex;
  flex-direction: column;
  gap: 1.25rem;
}

/* ── Header ── */
.site-header {
  text-align: center;
  padding: 1rem 0 0.5rem;
}

.logo {
  display: inline-flex;
  align-items: center;
  gap: 0.45rem;
  font-size: 1.75rem;
  font-weight: 700;
  letter-spacing: -0.5px;
  color: var(--text);
}

.logo-icon { font-size: 1.9rem; line-height: 1; }

.tagline {
  margin-top: 0.3rem;
  color: var(--muted);
  font-size: 0.93rem;
}

/* ── Card ── */
.card {
  background: var(--surface);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  box-shadow: var(--shadow);
  padding: 2rem;
}

/* ── Form ── */
.input-label {
  display: block;
  font-size: 0.88rem;
  font-weight: 600;
  color: var(--muted);
  text-transform: uppercase;
  letter-spacing: 0.04em;
  margin-bottom: 0.55rem;
}

.input-row {
  display: flex;
  gap: 0.6rem;
}

.url-input {
  flex: 1;
  padding: 0.7rem 1rem;
  border: 1.5px solid var(--border);
  border-radius: 7px;
  font-size: 0.97rem;
  font-family: "SFMono-Regular", "Consolas", monospace;
  color: var(--text);
  background: #fafbfc;
  transition: border-color var(--transition), box-shadow var(--transition);
  outline: none;
}

.url-input:focus {
  border-color: var(--accent);
  box-shadow: 0 0 0 3px rgba(59, 111, 212, 0.18);
  background: #fff;
}

.url-input.input-error-state {
  border-color: var(--danger);
  box-shadow: 0 0 0 3px rgba(192, 57, 43, 0.15);
}

.btn-analyse {
  padding: 0.7rem 1.4rem;
  background: var(--accent);
  color: #fff;
  border: none;
  border-radius: 7px;
  font-size: 0.97rem;
  font-weight: 600;
  cursor: pointer;
  white-space: nowrap;
  transition: background var(--transition), transform 0.1s;
  display: flex;
  align-items: center;
  gap: 0.4rem;
}

.btn-analyse:hover:not(:disabled) { background: var(--accent-hover); }
.btn-analyse:active:not(:disabled) { transform: scale(0.97); }
.btn-analyse:disabled { opacity: 0.65; cursor: not-allowed; }

/* Spinner */
.btn-spinner {
  width: 16px; height: 16px;
  border: 2px solid rgba(255,255,255,0.4);
  border-top-color: #fff;
  border-radius: 50%;
  animation: spin 0.7s linear infinite;
  display: inline-block;
}

@keyframes spin { to { transform: rotate(360deg); } }

.input-hint {
  margin-top: 0.45rem;
  font-size: 0.85rem;
  color: var(--danger);
  min-height: 1.2em;
}

/* ── Result panel ── */
.result-panel {
  margin-top: 1.75rem;
  border-top: 1px solid var(--border);
  padding-top: 1.75rem;
  animation: fadeIn 0.3s ease;
}

@keyframes fadeIn {
  from { opacity: 0; transform: translateY(8px); }
  to   { opacity: 1; transform: translateY(0); }
}

/* ── Score ring ── */
.score-section {
  display: flex;
  align-items: center;
  gap: 1.5rem;
  margin-bottom: 1.75rem;
  flex-wrap: wrap;
}

.score-ring-wrap {
  position: relative;
  width: 120px;
  height: 120px;
  flex-shrink: 0;
}

.score-ring {
  width: 120px;
  height: 120px;
  transform: rotate(-90deg);
}

.ring-bg {
  fill: none;
  stroke: var(--border);
  stroke-width: 10;
}

.ring-fill {
  fill: none;
  stroke: var(--safe-ring);
  stroke-width: 10;
  stroke-linecap: round;
  stroke-dasharray: 314;   /* 2π × 50 */
  stroke-dashoffset: 314;
  transition: stroke-dashoffset 0.7s cubic-bezier(0.4, 0, 0.2, 1),
              stroke 0.4s ease;
}

.score-centre {
  position: absolute;
  inset: 0;
  display: flex;
  flex-direction: column;
  align-items: center;
  justify-content: center;
}

.score-value {
  font-size: 2rem;
  font-weight: 700;
  line-height: 1;
  color: var(--text);
  transition: color 0.4s;
}

.score-label {
  font-size: 0.72rem;
  color: var(--muted);
  margin-top: 2px;
}

/* ── Verdict badge ── */
.verdict-block { flex: 1; }

.verdict-badge {
  display: inline-block;
  padding: 0.35rem 1rem;
  border-radius: 20px;
  font-size: 1rem;
  font-weight: 700;
  margin-bottom: 0.5rem;
  transition: background var(--transition), color var(--transition);
}

.band-safe .verdict-badge,
.verdict-badge.safe     { background: var(--safe-bg);       color: var(--safe); }
.verdict-badge.suspicious { background: var(--suspicious-bg); color: var(--suspicious); }
.verdict-badge.danger   { background: var(--danger-bg);     color: var(--danger); }

.verdict-url {
  word-break: break-all;
  font-family: "SFMono-Regular", "Consolas", monospace;
  font-size: 0.82rem;
  color: var(--muted);
}

/* ── Flags and improvements ── */
.section-title {
  font-size: 0.88rem;
  font-weight: 700;
  text-transform: uppercase;
  letter-spacing: 0.05em;
  color: var(--muted);
  margin-bottom: 0.65rem;
}

.flags-section,
.improvements-section {
  margin-bottom: 1.5rem;
}

.flags-list,
.improvements-list {
  list-style: none;
  display: flex;
  flex-direction: column;
  gap: 0.45rem;
}

.flags-list li {
  padding: 0.55rem 0.85rem;
  border-radius: 6px;
  background: var(--danger-bg);
  color: var(--danger);
  font-size: 0.9rem;
  border-left: 3px solid var(--danger);
}

.flags-list li::before { content: "⚠ "; }

.improvements-list li {
  padding: 0.55rem 0.85rem;
  border-radius: 6px;
  background: #eef4ff;
  color: #1a4a9e;
  font-size: 0.9rem;
  border-left: 3px solid var(--accent);
}

.improvements-list li::before { content: "💡 "; }

.no-flags {
  font-size: 0.9rem;
  color: var(--safe);
}

/* ── Band-aware ring and score colours ── */
.result-panel.band-safe      .ring-fill { stroke: var(--safe-ring); }
.result-panel.band-suspicious .ring-fill { stroke: var(--suspicious-ring); }
.result-panel.band-danger    .ring-fill { stroke: var(--danger-ring); }

.result-panel.band-safe      .score-value { color: var(--safe); }
.result-panel.band-suspicious .score-value { color: var(--suspicious); }
.result-panel.band-danger    .score-value { color: var(--danger); }

/* ── Scoring legend ── */
.scoring-legend {
  border: 1px solid var(--border);
  border-radius: 7px;
  overflow: hidden;
}

.scoring-legend summary {
  padding: 0.65rem 1rem;
  cursor: pointer;
  font-size: 0.88rem;
  font-weight: 600;
  color: var(--accent);
  user-select: none;
  list-style: none;
  display: flex;
  align-items: center;
  gap: 0.4rem;
}

.scoring-legend summary::before { content: "▶"; font-size: 0.7rem; }
.scoring-legend[open] summary::before { content: "▼"; }

.legend-body {
  padding: 1rem 1rem 1.1rem;
  border-top: 1px solid var(--border);
  background: #fafbfc;
}

.legend-body p { font-size: 0.88rem; color: var(--muted); margin-bottom: 0.8rem; }

.legend-table {
  width: 100%;
  border-collapse: collapse;
  font-size: 0.83rem;
}

.legend-table th,
.legend-table td {
  padding: 0.4rem 0.6rem;
  text-align: left;
  border-bottom: 1px solid var(--border);
}

.legend-table th {
  background: #f0f2f5;
  font-weight: 700;
  color: var(--muted);
  font-size: 0.78rem;
  text-transform: uppercase;
  letter-spacing: 0.04em;
}

.legend-bands {
  margin-top: 0.9rem !important;
  font-weight: 600;
  font-size: 0.85rem !important;
}

.band-safe        { color: var(--safe); }
.band-suspicious  { color: var(--suspicious); }
.band-danger      { color: var(--danger); }

/* ── Privacy note ── */
.privacy-note {
  text-align: center;
  font-size: 0.82rem;
  color: var(--muted);
}

/* ── Footer ── */
.site-footer {
  text-align: center;
  font-size: 0.78rem;
  color: #aab4c0;
  padding-top: 0.5rem;
  border-top: 1px solid var(--border);
}

/* ── Responsive ── */
@media (max-width: 500px) {
  .card { padding: 1.25rem; }
  .input-row { flex-direction: column; }
  .btn-analyse { width: 100%; justify-content: center; }
  .score-section { flex-direction: column; align-items: flex-start; }
}

In [ ]:
/**
 * PhishGuard — Frontend Interactions
 * ====================================
 * Handles form submission, API calls, animated score ring,
 * and result rendering. No URLs are stored client-side.
 */

"use strict";

// ── DOM references ──────────────────────────────────────────────────────────
const form          = document.getElementById("url-form");
const urlInput      = document.getElementById("url-input");
const btnAnalyse    = document.getElementById("btn-analyse");
const btnText       = btnAnalyse.querySelector(".btn-text");
const btnSpinner    = btnAnalyse.querySelector(".btn-spinner");
const inputError    = document.getElementById("input-error");

const resultPanel   = document.getElementById("result-panel");
const scoreValue    = document.getElementById("score-value");
const ringFill      = document.getElementById("ring-fill");
const verdictBadge  = document.getElementById("verdict-badge");
const verdictUrl    = document.getElementById("verdict-url");

const flagsList     = document.getElementById("flags-list");
const noFlags       = document.getElementById("no-flags");
const flagsSection  = document.getElementById("flags-section");

const improvList    = document.getElementById("improvements-list");
const noImprov      = document.getElementById("no-improvements");

// SVG ring circumference: 2π × r = 2π × 50 ≈ 314.16
const CIRCUMFERENCE = 2 * Math.PI * 50;

// ── Utility: validate URL format client-side (light check only) ─────────────
function looksLikeUrl(value) {
  return value.trim().length > 3 && /\S+\.\S+/.test(value.trim());
}

// ── UI helpers ───────────────────────────────────────────────────────────────
function setLoading(on) {
  btnAnalyse.disabled = on;
  btnText.hidden      = on;
  btnSpinner.hidden   = !on;
  urlInput.disabled   = on;
}

function showError(msg) {
  inputError.textContent = msg;
  urlInput.classList.add("input-error-state");
}

function clearError() {
  inputError.textContent = "";
  urlInput.classList.remove("input-error-state");
}

/**
 * Animate the SVG ring from its current dashoffset to the target value.
 * @param {number} score  0–100
 */
function animateRing(score) {
  const targetOffset = CIRCUMFERENCE - (score / 100) * CIRCUMFERENCE;
  // Use requestAnimationFrame for smooth start
  requestAnimationFrame(() => {
    ringFill.style.strokeDashoffset = targetOffset;
  });
}

/**
 * Animate the score counter from 0 to target.
 * @param {number} target
 */
function animateCounter(target) {
  const duration = 700; // ms
  const start    = performance.now();
  const from     = parseInt(scoreValue.textContent, 10) || 0;

  function step(now) {
    const elapsed  = now - start;
    const progress = Math.min(elapsed / duration, 1);
    // Ease-out cubic
    const eased    = 1 - Math.pow(1 - progress, 3);
    scoreValue.textContent = Math.round(from + (target - from) * eased);
    if (progress < 1) requestAnimationFrame(step);
    else scoreValue.textContent = target;
  }
  requestAnimationFrame(step);
}

// ── Render result ────────────────────────────────────────────────────────────
function renderResult(data) {
  const { score, verdict, band, url, flags, improvements } = data;

  // Panel visibility + band class
  resultPanel.hidden = false;
  resultPanel.className = "result-panel band-" + band;

  // Score ring + counter
  animateRing(score);
  animateCounter(score);

  // Verdict
  verdictBadge.textContent = verdict;
  verdictBadge.className   = "verdict-badge " + band;
  verdictUrl.textContent   = url;

  // Flags
  flagsList.innerHTML = "";
  if (flags && flags.length) {
    flags.forEach(f => {
      const li = document.createElement("li");
      li.textContent = f;
      flagsList.appendChild(li);
    });
    noFlags.hidden = true;
    flagsList.hidden = false;
  } else {
    noFlags.hidden = false;
    flagsList.hidden = true;
  }

  // Improvements
  improvList.innerHTML = "";
  if (improvements && improvements.length) {
    improvements.forEach(imp => {
      const li = document.createElement("li");
      li.textContent = imp;
      improvList.appendChild(li);
    });
    noImprov.hidden = true;
    improvList.hidden = false;
  } else {
    noImprov.hidden = false;
    improvList.hidden = true;
  }

  // Smooth scroll to result
  resultPanel.scrollIntoView({ behavior: "smooth", block: "nearest" });
}

// ── API call ─────────────────────────────────────────────────────────────────
async function analyseUrl(url) {
  const response = await fetch("/api/analyse", {
    method:  "POST",
    headers: { "Content-Type": "application/json" },
    body:    JSON.stringify({ url }),
  });

  if (!response.ok) {
    throw new Error(`Server returned ${response.status}`);
  }

  return response.json();
}

// ── Form submit handler ──────────────────────────────────────────────────────
form.addEventListener("submit", async (event) => {
  event.preventDefault();
  clearError();

  const raw = urlInput.value.trim();

  if (!raw) {
    showError("Please enter a URL.");
    urlInput.focus();
    return;
  }

  if (!looksLikeUrl(raw)) {
    showError("That doesn't look like a valid URL. Try including a domain, e.g. example.com");
    urlInput.focus();
    return;
  }

  setLoading(true);

  try {
    const data = await analyseUrl(raw);

    if (data.error) {
      showError("Analysis failed: " + data.error);
      return;
    }

    renderResult(data);
  } catch (err) {
    showError("Could not reach the server. Is Flask running?");
    console.error("Analysis error:", err);
  } finally {
    setLoading(false);
  }
});

// ── Clear error on re-type ───────────────────────────────────────────────────
urlInput.addEventListener("input", () => {
  if (inputError.textContent) clearError();
});

// ── Allow Enter key when input is focused (already handled by form submit) ──
// ── Keyboard shortcut: Ctrl/Cmd + Enter ─────────────────────────────────────
document.addEventListener("keydown", (e) => {
  if ((e.ctrlKey || e.metaKey) && e.key === "Enter") {
    form.requestSubmit();
  }
});